#Training Pipeline

In [27]:
import copy
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import numpy as np


In [2]:
# 1. SetUp & Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"using device: {device}")

using device: cpu


In [6]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [9]:
df.drop(columns = ['id', 'Unnamed: 32'], inplace=True)

In [10]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2) #Scikit-learn code
#This divides your dataset into two parts

##Scaling

In [ ]:
#The purpose is to bring your input features (columns) onto a similar scale so that a machine learning algorithm can learn more effectively.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#Why don't we use fit_transform() on X_test?
#This is a very important ML concept called Data Leakage.

#Imagine:
#- Training data has a mean CGPA of 7.5.
#- Testing data has a mean CGPA of 8.5.
#If you independently fit the scaler on testing data, you're using information from your test dataset to preprocess it differently.

#This creates an inconsistent evaluation process.

In [ ]:

X_train

array([[-0.41903369, -0.54087435, -0.45678044, ..., -0.22211322,
         0.58702003, -0.55641995],
       [ 0.55030901, -0.92832209,  0.58102053, ...,  1.28957731,
         0.18866346,  0.42518777],
       [-1.31588888,  0.37104534, -1.30210601, ..., -0.72818573,
        -0.03511454,  0.47279082],
       ...,
       [ 0.09264281,  0.07809704,  0.11446346, ...,  0.6581251 ,
        -0.28904701,  0.55048775],
       [ 0.38827812,  0.00958494,  0.41203784, ...,  0.58071148,
        -0.10018473, -0.17942568],
       [-1.30480256, -1.65360536, -1.29262649, ..., -0.77220524,
        -0.71438391, -0.3140274 ]])

In [17]:
y_train

454    B
330    M
66     B
213    M
550    B
      ..
50     B
32     M
99     M
131    M
21     B
Name: diagnosis, Length: 455, dtype: object

##Label Encoding


In [18]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


In [19]:
y_train

array([0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,

##Numpy array to Tensors 


In [24]:
X_train_tensors = torch.from_numpy(X_train)
X_test_tensors = torch.from_numpy(X_test)
y_train_tensors = torch.from_numpy(y_train)
y_test_tensors = torch.from_numpy(y_test)

In [30]:
X_train_tensors.shape

torch.Size([455, 30])

##Defining the model

In [ ]:
#You're creating a custom neural network class called mySimplenn using PyTorch.
#This class initializes the weights and bias of a simple neural network (or single linear neuron).
class mySimplenn():    #You're defining your own Python class. Think of it as a blueprint for creating a simple neural network.
    
    def __init__(self, X): #X represents your input training data.
        
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True) 
        #Creates a tensor containing random numbers between 0 and 1.
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        
    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

###Important Parameters 

In [ ]:
learning_rate = 0.1
epochs = x1 #How many runs on your data

##Training Model

In [ ]:
#Create Model
model = mySimplenn(X_train_tensors)

#Define loop
for epoch in range(epochs):
    #Forward Pass
    y_pred = model.forward(X_train_tensors)
    print(y_pred)